# OpenNeuro ECG/EMG Training - Local Storage Version
## No Google Drive Required

This version saves everything locally and provides download links at the end.

In [ ]:
# Setup without Google Drive
!pip install torch torchvision numpy pandas matplotlib scikit-learn
!pip install scipy neurokit2 heartpy biosppy
!pip install mne pyedflib
!pip install awscli boto3

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Create local directories
os.makedirs('/content/models', exist_ok=True)
os.makedirs('/content/checkpoints', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/processed_data', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Quick test with synthetic data to verify everything works
print("Creating synthetic ECG/EMG-like data for testing...")

np.random.seed(42)
n_samples = 200

# Create synthetic physiological features
synthetic_data = pd.DataFrame({
    # Heart rate features (from ECG)
    'mean_hr': np.random.normal(75, 10, n_samples),
    'std_hr': np.random.normal(5, 1, n_samples),
    'min_hr': np.random.normal(60, 5, n_samples),
    'max_hr': np.random.normal(90, 10, n_samples),
    'rmssd': np.random.normal(30, 10, n_samples),
    'pnn50': np.random.normal(15, 5, n_samples),
    'mean_rr': np.random.normal(800, 100, n_samples),
    'std_rr': np.random.normal(50, 10, n_samples),
    
    # EMG features
    'emg_mean_amplitude': np.random.normal(0.5, 0.1, n_samples),
    'emg_std_amplitude': np.random.normal(0.1, 0.02, n_samples),
    'emg_max_amplitude': np.random.normal(1.0, 0.2, n_samples),
    'emg_rms': np.random.normal(0.3, 0.05, n_samples),
    'emg_mean_frequency': np.random.normal(100, 20, n_samples),
    'emg_median_frequency': np.random.normal(90, 15, n_samples),
    'emg_zero_crossings': np.random.normal(0.1, 0.02, n_samples),
    
    # Label (binary classification)
    'label': np.random.randint(0, 2, n_samples)
})

print(f"Created synthetic dataset with {len(synthetic_data)} samples")
print(f"Features: {list(synthetic_data.columns[:-1])}")
synthetic_data.head()

In [ ]:
# Train/Test Split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and labels
X = synthetic_data.drop('label', axis=1).values
y = synthetic_data['label'].values

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Class distribution - Train: {np.bincount(y_train)}")
print(f"Class distribution - Test: {np.bincount(y_test)}")

In [ ]:
# Create PyTorch Dataset
class PhysioDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Create datasets and dataloaders
train_dataset = PhysioDataset(X_train_scaled, y_train)
test_dataset = PhysioDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"DataLoaders created: {len(train_loader)} train batches, {len(test_loader)} test batches")

In [ ]:
# Define the model
class ECGEMGNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32], num_classes=2, dropout=0.3):
        super(ECGEMGNet, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, num_classes))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

# Initialize model
input_dim = X_train_scaled.shape[1]
model = ECGEMGNet(input_dim=input_dim).to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Training function
def train_model(model, train_loader, test_loader, epochs=30):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    test_accuracies = []
    best_acc = 0
    
    for epoch in range(epochs):
        # Training
        model.train()
        total_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        # Evaluation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                _, predicted = outputs.max(1)
                total += batch_y.size(0)
                correct += predicted.eq(batch_y).sum().item()
        
        accuracy = 100. * correct / total
        test_accuracies.append(accuracy)
        
        # Save best model
        if accuracy > best_acc:
            best_acc = accuracy
            torch.save({
                'model_state_dict': model.state_dict(),
                'accuracy': accuracy,
                'epoch': epoch
            }, '/content/models/best_model.pth')
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}, Test Acc: {accuracy:.2f}%")
    
    print(f"\nTraining complete! Best accuracy: {best_acc:.2f}%")
    
    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(train_losses)
    ax1.set_title('Training Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    
    ax2.plot(test_accuracies)
    ax2.set_title('Test Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    
    plt.tight_layout()
    plt.savefig('/content/models/training_history.png')
    plt.show()
    
    return best_acc

# Train the model
best_accuracy = train_model(model, train_loader, test_loader, epochs=30)

In [ ]:
# Now try to download real data (optional)
print("Attempting to download real OpenNeuro data...")
print("This may take a while. You can skip this if you just want to test with synthetic data.")

!aws s3 ls --no-sign-request s3://openneuro.org/ds005873/ --recursive | head -20

# Download a sample
!aws s3 sync --no-sign-request s3://openneuro.org/ds005873 /content/data/ds005873 \
  --exclude "*" \
  --include "*.json" \
  --include "*.tsv" \
  --include "sub-001/*" \
  --max-concurrent-requests 5

print("\nDownloaded files:")
!find /content/data/ds005873 -type f | head -10

In [ ]:
# Save everything for download
import zipfile
import joblib

# Save the scaler
joblib.dump(scaler, '/content/models/scaler.pkl')

# Create a zip file with everything
with zipfile.ZipFile('/content/ecg_emg_model_package.zip', 'w') as zipf:
    # Add model files
    zipf.write('/content/models/best_model.pth', 'best_model.pth')
    zipf.write('/content/models/scaler.pkl', 'scaler.pkl')
    zipf.write('/content/models/training_history.png', 'training_history.png')
    
    # Create a simple inference script
    inference_code = f'''
import torch
import torch.nn as nn
import joblib
import numpy as np

class ECGEMGNet(nn.Module):
    def __init__(self, input_dim={input_dim}, hidden_dims=[64, 32], num_classes=2):
        super(ECGEMGNet, self).__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3)
            ])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

# Load model
model = ECGEMGNet()
checkpoint = torch.load('best_model.pth', map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Load scaler
scaler = joblib.load('scaler.pkl')

# Example inference
def predict(features):
    """features: numpy array of shape (n_features,) or (n_samples, n_features)"""
    features_scaled = scaler.transform(features.reshape(1, -1) if features.ndim == 1 else features)
    with torch.no_grad():
        outputs = model(torch.FloatTensor(features_scaled))
        predictions = outputs.argmax(dim=1)
    return predictions.numpy()

print(f"Model loaded! Accuracy: {{checkpoint['accuracy']:.2f}}%")
'''
    
    with zipf.open('inference.py', 'w') as f:
        f.write(inference_code.encode())

print("✅ Model package created: ecg_emg_model_package.zip")
print(f"📊 Best test accuracy: {best_accuracy:.2f}%")
print("\n📥 Download the package using the file browser on the left")
print("   or run: files.download('/content/ecg_emg_model_package.zip')")

# Provide download link
from google.colab import files
files.download('/content/ecg_emg_model_package.zip')